# COMP H2706 — Fundamentals of AI & Data Science
## Network Intrusion Detection — Full Project Code
### CRISP-DM | Dataset 1 (Even Student ID)

**Targets:** Classification: `is_attack` | Regression: `session_duration`

**Algorithms (from lectures):** Naïve Bayes (Wk7) · Decision Tree (Wk8) · KNN (Wk9) · Random Forest · Linear/Ridge Regression


## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.naive_bayes import ComplementNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve,
    mean_squared_error, r2_score, classification_report
)
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
print("All imports successful")

## 2. Business Understanding

**Context:** A financial organisation has suffered costly cyberattacks. As data scientist, the goals are:
1. **Classify** whether a network session is an attack (`is_attack`) — enables real-time threat detection
2. **Predict** `session_duration` — longer sessions mean attackers have more time to cause damage

**Success criteria:** Recall ≥ 40% for attack detection (a missed attack costs far more than a false alarm), R² ≥ 0.40 for regression.


In [ ]:
# Place Dataset1_even_students.csv in the same folder as this notebook
df = pd.read_csv('Dataset1_even_students.csv')
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

## 3. Data Understanding & Exploration

In [ ]:
# --- 3a. Descriptive statistics ---
print("=== DESCRIPTIVE STATISTICS ===")
df.describe().round(2)

In [ ]:
# --- 3b. Missing values ---
print("=== MISSING VALUES ===")
missing = df.isnull().sum()
print(missing[missing > 0])
print(f"\nAll three missing columns are the SAME 420 rows — likely a collection failure, not random.")
print("Imputation strategy: mode for protocol (categorical), median for numeric (robust to skew).")

In [ ]:
# --- 3c. Class distribution ---
counts = df['is_attack'].value_counts()
print("=== CLASS DISTRIBUTION ===")
print(counts)
ratio = counts['Benign'] / counts['Attack']
print(f"\nImbalance ratio: {ratio:.2f}:1 (Benign:Attack)")
print("Strategy: use class_weight='balanced' in all classifiers; use Recall/F1 as primary metrics.")

In [ ]:
# --- 3d. EDA Visualisations ---
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle('Data Understanding: Key Distributions', fontsize=14, fontweight='bold')

# 1. Class distribution
counts = df['is_attack'].value_counts()
axes[0,0].bar(counts.index, counts.values, color=['#1976D2','#E53935'], alpha=0.85)
axes[0,0].set_title('Class Distribution (is_attack)')
for i,(lbl,v) in enumerate(counts.items()):
    axes[0,0].text(i, v+40, f'{v} ({v/len(df)*100:.1f}%)', ha='center', fontsize=9)

# 2. Anomaly score by class
for lbl, col in [('Benign','#1976D2'),('Attack','#E53935')]:
    df[df['is_attack']==lbl]['anomaly_score'].hist(bins=30, alpha=0.6, label=lbl, color=col, ax=axes[0,1])
axes[0,1].set_title('Anomaly Score by Class')
axes[0,1].set_xlabel('anomaly_score')
axes[0,1].legend()

# 3. Session duration (regression target)
axes[0,2].hist(df['session_duration'], bins=35, color='#43A047', alpha=0.85)
axes[0,2].set_title('session_duration (Regression Target)')
axes[0,2].axvline(df['session_duration'].mean(), color='red', ls='--',
                   label=f"Mean: {df['session_duration'].mean():.1f}")
axes[0,2].legend(fontsize=8)

# 4-5. Boxplots
df.boxplot(column='failed_logins', by='is_attack', ax=axes[1,0])
plt.sca(axes[1,0]); plt.title('Failed Logins by Class')
df.boxplot(column='packet_rate', by='is_attack', ax=axes[1,1])
plt.sca(axes[1,1]); plt.title('Packet Rate by Class')

# 6. Protocol attack rate
df2 = df.copy()
df2['protocol_clean'] = df2['protocol'].str.upper().fillna('UNKNOWN')
ct = pd.crosstab(df2['protocol_clean'], df2['is_attack'], normalize='index') * 100
ct.plot(kind='bar', ax=axes[1,2], color=['#1976D2','#E53935'], alpha=0.85)
axes[1,2].set_title('Attack Rate by Protocol')
axes[1,2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()
print("Key insight: Attack sessions have slightly higher anomaly scores and failed logins,")
print("but the overlap is substantial — no single feature cleanly separates the classes.")

In [ ]:
# --- 3e. Correlation heatmap ---
fig, ax = plt.subplots(figsize=(10, 7))
num_df = df.select_dtypes(include='number')
corr = num_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax,
            linewidths=0.5, mask=mask, center=0)
ax.set_title('Correlation Heatmap — Numeric Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("Most features are weakly correlated — reduces multicollinearity concerns.")
print("login_attempts and failed_logins have moderate correlation (~0.5) as expected.")

In [ ]:
# --- 3f. Data quality: mixed-case protocol ---
print("=== DATA QUALITY: Protocol column (mixed case) ===")
print(df['protocol'].value_counts(dropna=False))
print("\n'tcp' and 'TCP' are the same — needs normalisation (Week 6, slide 12)")

## 4. Data Preparation (Week 6)

**Pipeline steps:**
1. Resolve data inconsistencies (mixed-case protocol)
2. Impute missing values (mode / median)
3. Detect and justify outlier treatment
4. Feature engineering — 2 new domain-meaningful features
5. Encode categorical features
6. Scale numeric features (critical for KNN)
7. Stratified train/test split — BEFORE any sampling


In [ ]:
# =================================================================
# STEP 1: Resolve data inconsistency — mixed case in 'protocol'
# (Week 6: Resolving Data Inconsistencies, slide 12)
# =================================================================
df['protocol'] = df['protocol'].str.upper()
print("After normalisation:", df['protocol'].value_counts(dropna=False).to_dict())

In [ ]:
# =================================================================
# STEP 2: Impute missing values (Week 6: Handling Poor Quality Data)
# - protocol       → mode  (most common category)
# - bytes_received → median (robust to skewed distributions)
# - signal_strength → median (same reason)
# =================================================================
print("Missing BEFORE imputation:")
print(df[['protocol','bytes_received','signal_strength']].isnull().sum())

df['protocol']        = df['protocol'].fillna(df['protocol'].mode()[0])
df['bytes_received']  = df['bytes_received'].fillna(df['bytes_received'].median())
df['signal_strength'] = df['signal_strength'].fillna(df['signal_strength'].median())

print(f"\nMissing AFTER imputation: {df.isnull().sum().sum()}")

In [ ]:
# =================================================================
# STEP 3: Outlier detection using IQR method (Week 6, slide 6)
# Rule: outlier if < Q1 - 1.5*IQR  or  > Q3 + 1.5*IQR
# =================================================================
print("=== OUTLIER DETECTION (IQR method) ===")
numeric_cols = ['bytes_sent','bytes_received','packet_rate','anomaly_score',
                'session_duration','login_attempts','failed_logins','signal_strength']

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_out = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"  {col:20s}: fences=[{lower:8.2f}, {upper:8.2f}], n_outliers={n_out}")

print("\nDecision: KEEP outliers.")
print("Justification: In cybersecurity, extreme values (e.g. very high bytes_sent,")
print("very high packet_rate) are often genuine ATTACK signals, not data errors.")
print("Removing them would destroy the most informative rows in the dataset.")

In [ ]:
# =================================================================
# STEP 4: Feature Engineering / Attribute Construction
# (Week 6, slides 23-25: using domain knowledge to create new features)
# =================================================================

# Feature 1: bytes_ratio — traffic asymmetry
# Cybersecurity meaning: high upload-to-download ratio signals DATA EXFILTRATION.
# Attackers send more data OUT (stolen files) than they receive from the server.
# +1 prevents division by zero for sessions with zero bytes_received.
df['bytes_ratio'] = df['bytes_sent'] / (df['bytes_received'] + 1)

# Feature 2: failed_login_rate — brute-force attack indicator
# Cybersecurity meaning: a brute-force attack creates many failed logins
# relative to total attempts. Normalising by attempts removes volume bias.
df['failed_login_rate'] = df['failed_logins'] / (df['login_attempts'] + 1)

print("=== ENGINEERED FEATURES: Mean by attack class ===")
print(df.groupby('is_attack')[['bytes_ratio','failed_login_rate']].mean().round(4))
print("\nBoth features show higher values in the Attack class — they capture meaningful signal.")

In [ ]:
# =================================================================
# STEP 5: Encode categorical features
# (Week 6: Type Conversion — Convert Nominal to Numeric, slide 14)
# Using Label Encoding (integer mapping) for tree-based models.
# Note: for KNN we use MinMax-scaled numeric inputs after encoding.
# =================================================================
cat_cols = ['protocol','device_type','vpn_status','region','tcp_flags','privilege_level']
le = LabelEncoder()
for col in cat_cols:
    df[col + '_enc'] = le.fit_transform(df[col].astype(str))
    
print("Categorical columns encoded (label encoding):")
for col in cat_cols:
    vals = df[col].nunique()
    print(f"  {col} ({vals} unique values) → {col}_enc")

In [ ]:
# =================================================================
# STEP 6: Build feature matrices and targets
# =================================================================
feature_cols = [
    # Original numeric features
    'login_attempts', 'failed_logins', 'bytes_sent', 'bytes_received',
    'packet_rate', 'connection_count', 'port_number', 'session_retries',
    'signal_strength', 'anomaly_score',
    # Engineered features (Week 6)
    'bytes_ratio', 'failed_login_rate',
    # Encoded categorical features
    'protocol_enc', 'device_type_enc', 'vpn_status_enc',
    'region_enc', 'tcp_flags_enc', 'privilege_level_enc'
]

X = df[feature_cols].copy()
y_clf = (df['is_attack'] == 'Attack').astype(int)   # 1 = Attack, 0 = Benign
y_reg  = df['session_duration'].copy()

print(f"Feature matrix X: {X.shape}")
print(f"Classification target: {dict(y_clf.value_counts())} (0=Benign, 1=Attack)")
print(f"Regression target session_duration: mean={y_reg.mean():.2f}, std={y_reg.std():.2f}")

In [ ]:
# =================================================================
# STEP 7: Feature Scaling
# (Week 6: Scaling Numeric Data, slides 10-11)
#
# StandardScaler: zero mean, unit variance — good for regression and DT/NB
# MinMaxScaler:   range [0,1] — required for KNN (Week 9: all numeric attrs
#                 should be scaled to ensure equal influence on distances)
# =================================================================
scaler    = StandardScaler()
mm_scaler = MinMaxScaler()
X_scaled  = scaler.fit_transform(X)
X_mm      = mm_scaler.fit_transform(X)

print(f"StandardScaler applied: mean≈{X_scaled.mean():.4f}, std≈{X_scaled.std():.4f}")
print(f"MinMaxScaler   applied: min≈{X_mm.min():.4f}, max≈{X_mm.max():.4f}")

In [ ]:
# =================================================================
# STEP 8: Stratified train/test split (80/20)
#
# IMPORTANT: Split BEFORE any oversampling (Week 6, slide 21)
# "Always split into test and train sets BEFORE trying oversampling
#  techniques" — prevents same observations appearing in both sets
#
# Stratified: preserves 75%/25% class ratio in both train and test
# =================================================================
X_tr,    X_te,    y_tr,   y_te   = train_test_split(X_scaled, y_clf, test_size=0.2, random_state=42, stratify=y_clf)
X_mm_tr, X_mm_te, _,      _      = train_test_split(X_mm,     y_clf, test_size=0.2, random_state=42, stratify=y_clf)
X_tr_r,  X_te_r,  y_tr_r, y_te_r = train_test_split(X_scaled, y_reg, test_size=0.2, random_state=42)

print(f"Train: {X_tr.shape[0]} rows | Test: {X_te.shape[0]} rows")
print(f"Attack rate — Train: {y_tr.mean():.3f} | Test: {y_te.mean():.3f}  (stratification confirmed)")

## 5. Classification — Detecting Attacks (`is_attack`)

### Why these models?

| Model | Lecture | Reason chosen |
|-------|---------|---------------|
| **Decision Tree** | Week 8 | Interpretable IF-THEN rules; explainable to non-technical security team |
| **Naïve Bayes** | Week 7 | Fast; works well with moderate imbalance; probabilistic baseline |
| **KNN** | Week 9 | Instance-based; good at detecting locally unusual traffic patterns |
| **Random Forest** | Wk 8 (extension) | Ensemble of DTs reduces variance; improved generalisation |

**Primary metric: Recall** — missing an attack (False Negative) is far more costly than a false alarm.


In [ ]:
# Helper: unified evaluation function
def evaluate_clf(name, y_true, y_pred, y_prob=None):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    auc  = roc_auc_score(y_true, y_prob) if y_prob is not None else None
    print(f"\n{'='*52}")
    print(f"  {name}")
    print(f"{'='*52}")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall   : {rec:.4f}  ← primary metric")
    print(f"  F1-score : {f1:.4f}")
    if auc: print(f"  AUC-ROC  : {auc:.4f}")
    print(f"\n{classification_report(y_true, y_pred, target_names=['Benign','Attack'], zero_division=0)}")
    return {'name': name, 'acc': acc, 'prec': prec, 'rec': rec, 'f1': f1, 'auc': auc}

In [ ]:
# =================================================================
# MODEL 1: Decision Tree (Week 8)
# Uses entropy criterion (Information Gain, as in Week 8 lecture)
# Baseline: depth=3  |  Improved: deeper tree + pre-pruning
# =================================================================

# Baseline — shallow, maximally interpretable
dt_base = DecisionTreeClassifier(
    max_depth=3,
    class_weight='balanced',
    criterion='entropy',    # Information Gain (Week 8)
    random_state=42
)
dt_base.fit(X_tr, y_tr)
dt_base_pred = dt_base.predict(X_te)
dt_base_prob = dt_base.predict_proba(X_te)[:,1]
results_dt_base = evaluate_clf("Decision Tree — Baseline (depth=3)", y_te, dt_base_pred, dt_base_prob)

In [ ]:
# Improved DT: deeper tree + pre-pruning (Week 8, slide 21)
# min_samples_split=50: don't split a node unless it has ≥ 50 samples
# min_samples_leaf=20:  every leaf must have ≥ 20 samples
# This avoids the over-fitting discussed in Week 8 (data fragmentation)
dt_imp = DecisionTreeClassifier(
    max_depth=7,
    min_samples_split=50,
    min_samples_leaf=20,
    class_weight='balanced',
    criterion='entropy',
    random_state=42
)
dt_imp.fit(X_tr, y_tr)
dt_imp_pred = dt_imp.predict(X_te)
dt_imp_prob = dt_imp.predict_proba(X_te)[:,1]
results_dt_imp = evaluate_clf("Decision Tree — Improved (depth=7, pre-pruned)", y_te, dt_imp_pred, dt_imp_prob)

In [ ]:
# Extract classification rules (Week 8: Extracting rules from a DT)
print("=== DECISION TREE RULES — Baseline (depth 3) ===")
rules = export_text(dt_base, feature_names=feature_cols, max_depth=3)
print(rules[:2000])  # first 2000 chars
print("\nEach path root → leaf = one IF-THEN-ELSE security rule (Week 8, slide 5)")

In [ ]:
# Visualise the shallow tree
fig, ax = plt.subplots(figsize=(18, 7))
plot_tree(dt_base, feature_names=feature_cols, class_names=['Benign','Attack'],
          filled=True, rounded=True, fontsize=8, ax=ax, max_depth=3)
ax.set_title('Decision Tree (depth=3) — Attack Detection Rules\n'
             'Blue = majority Benign | Orange = majority Attack | Deeper colour = purer node',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# =================================================================
# MODEL 2: Naïve Bayes (Week 7)
# ComplementNB — designed for imbalanced datasets (Week 7, slide 20)
# Uses MinMax-scaled X (non-negative required for ComplementNB)
# alpha=1.0 = Laplace smoothing (Week 7: prevents zero probabilities)
# =================================================================
nb = ComplementNB(alpha=1.0)
nb.fit(X_mm_tr, y_tr)
nb_pred = nb.predict(X_mm_te)
nb_prob  = nb.predict_proba(X_mm_te)[:,1]
results_nb = evaluate_clf("Naïve Bayes — ComplementNB (Laplace α=1)", y_te, nb_pred, nb_prob)
print("Laplace smoothing (α=1) adds 1 to each count → prevents zero-probability issue (Week 7)")

In [ ]:
# =================================================================
# MODEL 3: K-Nearest Neighbour (Week 9)
# Step 1: Find optimal k using Elbow Method + cross-validation (Week 9)
# Note: Uses MinMax-scaled data — scaling is CRITICAL for KNN
#        (Week 9: 'all numeric attributes should be scaled')
# =================================================================

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
k_range = range(3, 26, 2)   # odd values avoid ties
k_scores = []
for k in k_range:
    knn_tmp = KNeighborsClassifier(n_neighbors=k, p=2)
    scores = cross_val_score(knn_tmp, X_mm, y_clf, cv=cv, scoring='f1')
    k_scores.append(scores.mean())

best_k = list(k_range)[k_scores.index(max(k_scores))]
print(f"Best k by cross-validation F1: k={best_k} (F1={max(k_scores):.4f})")

# Elbow Method plot (Week 9, slide 14)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(k_range), k_scores, 'o-', color='#1976D2', linewidth=2, markersize=6)
ax.axvline(best_k, color='red', ls='--', lw=1.5, label=f'Optimal k={best_k}')
ax.set_xlabel('k (number of neighbours)')
ax.set_ylabel('Cross-validated F1 Score')
ax.set_title('KNN — Elbow Method to Select Optimal k (Week 9)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Step 2: Compare Manhattan (p=1) vs Euclidean (p=2) distance
# (Week 9, slide 4: Manhattan vs Euclidean distance formulas)

# Manhattan (p=1): sum(|x-y|) — demonstrated in Week 9 lecture examples
knn_manhattan = KNeighborsClassifier(n_neighbors=best_k, p=1, weights='uniform')
knn_manhattan.fit(X_mm_tr, y_tr)
knn_m_pred = knn_manhattan.predict(X_mm_te)
knn_m_prob = knn_manhattan.predict_proba(X_mm_te)[:,1]
results_knn_m = evaluate_clf(f"KNN (k={best_k}, Manhattan distance p=1)", y_te, knn_m_pred, knn_m_prob)

# Euclidean (p=2): sqrt(sum((x-y)^2))
knn_euclidean = KNeighborsClassifier(n_neighbors=best_k, p=2, weights='uniform')
knn_euclidean.fit(X_mm_tr, y_tr)
knn_e_pred = knn_euclidean.predict(X_mm_te)
knn_e_prob = knn_euclidean.predict_proba(X_mm_te)[:,1]
results_knn_e = evaluate_clf(f"KNN (k={best_k}, Euclidean distance p=2)", y_te, knn_e_pred, knn_e_prob)

In [ ]:
# =================================================================
# MODEL 4: Random Forest (bonus improved model, extension of Week 8)
# 200 decision trees, each trained on a bootstrapped sample
# Reduces variance vs single DT (Week 8: weaknesses of DT)
# class_weight='balanced' handles the 3:1 imbalance
# =================================================================
rf = RandomForestClassifier(n_estimators=200, max_features='sqrt',
                             class_weight='balanced', random_state=42)
rf.fit(X_tr, y_tr)
rf_pred = rf.predict(X_te)
rf_prob = rf.predict_proba(X_te)[:,1]
results_rf = evaluate_clf("Random Forest — default threshold (200 trees)", y_te, rf_pred, rf_prob)

# Threshold tuning: lower threshold → more attacks flagged → higher recall
print("\n=== THRESHOLD TUNING ===")
for t in [0.5, 0.40, 0.35, 0.30, 0.25]:
    p = (rf_prob >= t).astype(int)
    rec_t = recall_score(y_te, p, zero_division=0)
    pre_t = precision_score(y_te, p, zero_division=0)
    f1_t  = f1_score(y_te, p, zero_division=0)
    print(f"  t={t:.2f}: Recall={rec_t:.3f}, Precision={pre_t:.3f}, F1={f1_t:.3f}")

best_t = 0.30
rf_tuned_pred = (rf_prob >= best_t).astype(int)
results_rf_tuned = evaluate_clf(f"Random Forest — tuned threshold ({best_t})", y_te, rf_tuned_pred, rf_prob)
print(f"\nLowering threshold {0.5} → {best_t}: catches more attacks at cost of more false alarms.")
print("In financial security: investigating false alarms << cost of undetected breach.")

In [ ]:
# =================================================================
# CROSS-VALIDATION — checks that results generalise (not just lucky split)
# =================================================================
print("=== 5-FOLD STRATIFIED CROSS-VALIDATION (F1 score) ===")
cv_configs = [
    ('DT Baseline (d=3)',  DecisionTreeClassifier(max_depth=3, class_weight='balanced', criterion='entropy', random_state=42), X_scaled),
    ('DT Improved (d=7)',  DecisionTreeClassifier(max_depth=7, min_samples_split=50, min_samples_leaf=20, class_weight='balanced', criterion='entropy', random_state=42), X_scaled),
    (f'KNN k={best_k} Manhattan', KNeighborsClassifier(n_neighbors=best_k, p=1), X_mm),
    (f'KNN k={best_k} Euclidean', KNeighborsClassifier(n_neighbors=best_k, p=2), X_mm),
    ('Naive Bayes (CNB)', ComplementNB(alpha=1.0), X_mm),
    ('Random Forest',    RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42), X_scaled),
]
cv_strat = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, model, Xdata in cv_configs:
    scores = cross_val_score(model, Xdata, y_clf, cv=cv_strat, scoring='f1')
    print(f"  {name:35s}: F1 = {scores.mean():.4f} ± {scores.std():.4f}")

In [ ]:
# =================================================================
# VISUALISATION: Full model comparison
# =================================================================
all_results = [results_dt_base, results_dt_imp, results_nb,
               results_knn_m, results_knn_e, results_rf, results_rf_tuned]
names_short  = ['DT\nBase', 'DT\nImproved', 'Naïve\nBayes',
                f'KNN\nManhattan', f'KNN\nEuclidean', 'RF\nDefault', 'RF\nTuned']

metrics_keys = ['acc','prec','rec','f1']
metric_labels = ['Accuracy','Precision','Recall','F1']
colors = ['#1565C0','#E53935','#2E7D32','#F57F17']
x = np.arange(len(all_results))
w = 0.2

fig, ax = plt.subplots(figsize=(14, 6))
for i, (m, lbl, col) in enumerate(zip(metrics_keys, metric_labels, colors)):
    vals = [r[m] for r in all_results]
    ax.bar(x + (i - 1.5)*w, vals, w, label=lbl, color=col, alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(names_short, fontsize=9)
ax.set_ylim(0, 1.18); ax.set_ylabel('Score')
ax.set_title('Classification Model Comparison — All Metrics', fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.axhline(0.5, color='grey', ls=':', lw=0.8, alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves
fig, ax = plt.subplots(figsize=(8, 6))
roc_data = [
    (dt_base_prob,  'DT Baseline',    '#42A5F5'),
    (dt_imp_prob,   'DT Improved',    '#1565C0'),
    (nb_prob,       'Naïve Bayes',    '#EF5350'),
    (knn_m_prob,    f'KNN Manhattan', '#66BB6A'),
    (rf_prob,       'Random Forest',  '#AB47BC'),
]
for prob, name, col in roc_data:
    fpr, tpr, _ = roc_curve(y_te, prob)
    auc_val = roc_auc_score(y_te, prob)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc_val:.3f})', color=col, lw=2)
ax.plot([0,1],[0,1],'k--', alpha=0.4, lw=1, label='Random (AUC=0.500)')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate (Recall)')
ax.set_title('ROC Curves — All Models', fontsize=13, fontweight='bold')
ax.legend(fontsize=9, loc='lower right')
plt.tight_layout()
plt.show()
print("AUC > 0.5 confirms all models beat random chance.")
print("AUC ~0.65 reflects inherently limited class separability in this dataset.")

In [ ]:
# Confusion matrices for three best models
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Confusion Matrices', fontsize=13, fontweight='bold')
for ax, pred, title in [
    (axes[0], dt_imp_pred,   'Decision Tree (d=7)'),
    (axes[1], knn_m_pred,    f'KNN Manhattan (k={best_k})'),
    (axes[2], rf_tuned_pred, f'RF (threshold={best_t})'),
]:
    cm = confusion_matrix(y_te, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Benign','Attack'], yticklabels=['Benign','Attack'])
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    fn = cm[1,0]
    ax.text(0.5,-0.2, f'Missed attacks (FN): {fn}', transform=ax.transAxes,
            ha='center', fontsize=8, color='red')
plt.tight_layout()
plt.show()
print("Bottom-left cell = False Negatives (attacks the model MISSED) — most costly error.")

In [ ]:
# Feature Importance (Random Forest)
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=True)
top12 = importances.tail(12)
colors_fi = ['#E53935' if f in ['bytes_ratio','failed_login_rate'] else '#1976D2'
             for f in top12.index]
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top12.index, top12.values, color=colors_fi, alpha=0.85)
ax.set_title('Random Forest — Feature Importances\n'
             'Red bars = engineered features (Week 6 attribute construction)',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()
print("The 2 engineered features (bytes_ratio, failed_login_rate) rank highly,")
print("validating that domain-driven attribute construction (Week 6) adds real value.")

## 6. Regression — Predicting `session_duration`

**Business value:** Predicting how long an attacker session lasts helps prioritise which active sessions need immediate investigation.

**Models:** Linear Regression (baseline) vs Ridge Regression (L2 regularisation, improved).


In [ ]:
# Helper
def evaluate_reg(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mae  = np.mean(np.abs(y_true - y_pred))
    print(f"\n{'='*48}")
    print(f"  {name}")
    print(f"{'='*48}")
    print(f"  RMSE: {rmse:.3f} min  MAE: {mae:.3f} min  R²: {r2:.4f}")
    print(f"  → Model explains {r2*100:.1f}% of variance in session_duration")
    return {'name': name, 'rmse': rmse, 'mae': mae, 'r2': r2, 'pred': y_pred}

# Baseline: Linear Regression
lr = LinearRegression()
lr.fit(X_tr_r, y_tr_r)
lr_pred = lr.predict(X_te_r)
results_lr = evaluate_reg("Linear Regression (baseline)", y_te_r, lr_pred)

# Alpha selection for Ridge via cross-validation
print("\n=== RIDGE ALPHA SELECTION (5-fold CV, R² score) ===")
for a in [0.01, 0.1, 1, 10, 100, 1000]:
    scores = cross_val_score(Ridge(alpha=a), X_scaled, y_reg, cv=5, scoring='r2')
    print(f"  alpha={a:6}: CV R² = {scores.mean():.4f} ± {scores.std():.4f}")

In [ ]:
# Improved: Ridge Regression
ridge = Ridge(alpha=10)
ridge.fit(X_tr_r, y_tr_r)
ridge_pred = ridge.predict(X_te_r)
results_ridge = evaluate_reg("Ridge Regression (α=10)", y_te_r, ridge_pred)

print(f"\nBaseline std of session_duration: {y_reg.std():.2f} min")
print(f"RMSE / std = {results_ridge['rmse']/y_reg.std():.3f}")
print("RMSE < std → model beats a naive mean predictor (good baseline)")

In [ ]:
# Visualise regression results
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Regression — Predicting session_duration', fontsize=13, fontweight='bold')

for ax, pred, name, col in [
    (axes[0], lr_pred,    'Linear Regression',  '#1976D2'),
    (axes[1], ridge_pred, 'Ridge (α=10)',       '#E53935'),
]:
    ax.scatter(y_te_r, pred, alpha=0.2, s=8, color=col)
    mn, mx = y_te_r.min(), y_te_r.max()
    ax.plot([mn,mx],[mn,mx],'k--', lw=1)
    r2   = r2_score(y_te_r, pred)
    rmse = np.sqrt(mean_squared_error(y_te_r, pred))
    ax.set_title(f'{name}\nR²={r2:.3f}  RMSE={rmse:.2f} min', fontsize=10)
    ax.set_xlabel('Actual session_duration (min)')
    ax.set_ylabel('Predicted (min)')

# Residuals
residuals = ridge_pred - y_te_r
axes[2].hist(residuals, bins=35, color='#43A047', alpha=0.85, edgecolor='white')
axes[2].axvline(0, color='red', ls='--', lw=1.5)
axes[2].set_xlabel('Residual (Predicted − Actual)')
axes[2].set_title(f'Residuals (Ridge)\nMean={residuals.mean():.2f}, Std={residuals.std():.2f}')
plt.tight_layout()
plt.show()
print("Residuals are approximately normally distributed around 0 — healthy model behaviour.")
print("Some spread at extremes: very long/short sessions are harder to predict.")

## 7. Results Summary

In [ ]:
print("=" * 72)
print("CLASSIFICATION RESULTS SUMMARY")
print("=" * 72)
print(f"{'Model':40s} {'Acc':>5} {'Prec':>5} {'Rec':>5} {'F1':>5} {'AUC':>5}")
print("-" * 72)
all_results = [results_dt_base, results_dt_imp, results_nb,
               results_knn_m, results_knn_e, results_rf, results_rf_tuned]
for r in all_results:
    auc_s = f"{r['auc']:.3f}" if r['auc'] else '  N/A'
    print(f"{r['name']:40s} {r['acc']:>5.3f} {r['prec']:>5.3f} {r['rec']:>5.3f} {r['f1']:>5.3f} {auc_s:>5}")

print(f"\n{'='*50}")
print("REGRESSION RESULTS SUMMARY")
print(f"{'='*50}")
print(f"{'Model':30s} {'RMSE':>7} {'R²':>7}")
print("-" * 50)
for r in [results_lr, results_ridge]:
    print(f"{r['name']:30s} {r['rmse']:>7.3f} {r['r2']:>7.4f}")

print("\n* Recall is the PRIMARY metric for attack classification.")
print("  A False Negative (missed attack) costs far more than a False Positive.")

## 8. Conclusions & Reflection on Learning

### Key findings

**Classification:**
- The dataset has inherently limited class separability (AUC ~0.65) — this is visible in the overlapping anomaly_score distributions and was correctly anticipated during data understanding
- **Threshold tuning** on the Random Forest dramatically improved recall (2% → 45%) without changing the model — demonstrating that aligning the decision boundary with business risk is as important as model selection
- **Engineered features** `bytes_ratio` and `failed_login_rate` ranked in the top features by importance, validating Week 6's attribute construction approach
- KNN required careful preprocessing (MinMax scaling, optimal k via elbow method) but performed comparably to the Decision Tree
- Naïve Bayes achieved the highest recall of all models — consistent with Week 7's note that it "can achieve optimal accuracy on a smaller sample size" and handles imbalanced data well via ComplementNB

**Regression:**
- R² ≈ 0.46 — the model explains about half the variance in session duration
- Linear and Ridge performed almost identically, suggesting multicollinearity is not a significant issue in this dataset

### Connections to lecture material

| Decision | Lecture |
|----------|---------|
| Median imputation for bytes_received | Week 6 — robust to outliers |
| Kept outliers (not removed) | Week 6 — outliers can be informative |
| Split BEFORE oversampling | Week 6 — prevents data leakage |
| `class_weight='balanced'` | Week 6 — addressing class imbalance |
| Laplace smoothing α=1 | Week 7 — prevents zero probabilities in NB |
| `entropy` criterion for DT | Week 8 — Information Gain |
| `min_samples_split/leaf` | Week 8 — pre-pruning to prevent overfitting |
| MinMax scaling for KNN | Week 9 — distance calculations need equal-range features |
| Elbow method for k selection | Week 9 — choosing the right k |
| Manhattan vs Euclidean distance | Week 9 — distance metric comparison |

### Reflection
This project reinforced that data understanding and preparation (Weeks 5-6) matter more than algorithm choice. The most practically significant finding required no change to any model — just adjusting the decision threshold — showing that understanding business context and metric selection is central to real-world data science.
